# Real-image object replacement — walkthrough

Single source of truth for the pipeline. Top-to-bottom execution produces:
- a reconstruction-LPIPS table per image (the gate for whether an image is usable)
- a per-schedule edit grid for demo
- a full ablation CSV across 5 schedules x 2 mask modes
- the headline figures for the report

**Robustness design:** real-photo prompts are never going to be perfect. Rather than
hand-curating prompts for every image, we (a) auto-caption each photo with BLIP so the
source prompt actually matches what's in frame, (b) gate every image on a reconstruction
LPIPS check before editing it, (c) skip any image whose reconstruction drifts beyond a
threshold even at bumped `inner_steps`, and (d) always apply the pixel composite so
background is bit-exact regardless of how the edit went inside the mask.

## 1. Setup

One-time per Colab runtime. If the repo's already cloned this is mostly a no-op.

In [ ]:
import os
from pathlib import Path

if not Path('/content/object_replace').exists():
    !git clone https://github.com/stefan-arni/object_replace.git /content/object_replace
%cd /content/object_replace
!git pull
!pip install -q -r requirements.txt

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, 'src')

import torch
from PIL import Image
import matplotlib.pyplot as plt

from sd_components import load_sd, decode_latents, encode_prompt
from null_text_inv import null_text_inversion, sample_with_null
from editor import Editor, _to_pil
from schedules import (
    vanilla_p2p, linear_decay_replaced, cosine_replaced,
    constant_replaced, piecewise_demo,
)
from metrics import reconstruction_lpips, clip_directional_similarity, background_lpips
from masks import visualize_mask

print('loading Stable Diffusion components...')
c = load_sd()
editor = Editor(c)
print(f'device: {c.device}, dtype: {c.dtype}')

In [ ]:
from transformers import BlipForConditionalGeneration, BlipProcessor

print('loading BLIP (used for auto-captioning source prompts)...')
_blip_proc = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
_blip_model = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base'
).to(c.device).eval()

@torch.no_grad()
def caption(pil_img):
    inputs = _blip_proc(pil_img, return_tensors='pt').to(c.device)
    out = _blip_model.generate(**inputs, max_new_tokens=30)
    return _blip_proc.decode(out[0], skip_special_tokens=True).strip()

## 2. Reconstruction gate

The gate: given an image and a source prompt, run null-text inversion + re-sample, measure
reconstruction LPIPS. If above `lpips_threshold`, retry with more `inner_steps`. If still bad,
this image gets skipped from the ablation -- its edit would be garbage regardless of schedule.

In [ ]:
def reconstruct_and_gate(img, source_prompt, lpips_threshold=0.15, steps=50, guidance=7.5, verbose=True):
    """Returns (recon_lpips, passed: bool, inner_steps_used).

    0.15 threshold (was 0.10): the null-text paper itself reports ~0.05-0.15
    on their real-photo benchmark. Real photos inherently drift more than the
    generated images we tested with during development.
    """
    for inner_steps in (10, 20):
        nt = null_text_inversion(
            c, img, source_prompt,
            num_inference_steps=steps, guidance_scale=guidance,
            inner_steps=inner_steps, verbose=False,
        )
        cond = encode_prompt(c, source_prompt)
        z0 = sample_with_null(c, cond, nt.null_embeds, nt.z_T, steps, guidance)
        recon = _to_pil(decode_latents(c, z0).clamp(-1, 1))
        lpips_score = reconstruction_lpips(img, recon)
        if verbose:
            print(f'  inner_steps={inner_steps}: LPIPS={lpips_score:.4f}', end='')
        if lpips_score < lpips_threshold:
            if verbose:
                print('  [PASS]')
            return lpips_score, True, inner_steps
        if verbose:
            print(f'  [above {lpips_threshold}, retrying]' if inner_steps == 10 else '  [FAIL]')
    return lpips_score, False, inner_steps

## 3. Dataset inventory + prompt generation

If `data/prompts.json` doesn't exist, we auto-fetch the COCO subset and caption it.
Otherwise we trust what's already there (you can edit prompts.json by hand to fix
BLIP's mistakes).

In [ ]:
real_dir = Path('data/real')
prompts_path = Path('data/prompts.json')

# Bootstrap: if prompts.json doesn't exist, run the fetch script which both
# downloads COCO images and writes the initial (bare) prompts.json.
# (Checking real_dir isn't reliable -- a .gitkeep file makes it "non-empty".)
if not prompts_path.exists():
    print('no prompts.json yet; fetching COCO subset + writing bare prompts...')
    !python scripts/fetch_real_photos.py

# If prompts still look auto-generated bare ('a photograph of a X'), upgrade
# them with BLIP captions. Real-photo inversion needs descriptive prompts;
# bare ones drift hard.
_entries = json.loads(prompts_path.read_text())
if _entries and _entries[0]['source'].startswith('a photograph of a '):
    print('upgrading bare prompts to BLIP captions...')
    !python scripts/caption_real_photos.py

entries = json.loads(prompts_path.read_text())
print(f'{len(entries)} entries in prompts.json')
for e in entries[:5]:
    print(f'  {e["image"]}: {e["source"][:50]!r} -> {e["target"][:50]!r}')

## 4. Single-image demo

Pick one image (defaults to the first in prompts.json). Show the source, the BLIP
caption, the reconstruction, and five schedule outputs with+without mask.

In [ ]:
DEMO_IDX = 0  # change to pick a different image

entry = entries[DEMO_IDX]
img = Image.open(real_dir / entry['image']).convert('RGB').resize((512, 512))
src_prompt = entry['source']
tgt_prompt = entry['target']
print(f'image:  {entry["image"]}')
print(f'source: {src_prompt!r}')
print(f'target: {tgt_prompt!r}')
img

In [ ]:
print('reconstruction gate:')
lpips_score, passed, inner_used = reconstruct_and_gate(img, src_prompt)
if not passed:
    print(f'  FAIL (LPIPS={lpips_score:.4f}). This image would be skipped in the ablation.')
else:
    print(f'  PASS using inner_steps={inner_used}')

In [ ]:
SCHEDULES = {
    'vanilla_p2p':   vanilla_p2p(0.8),
    'linear_decay':  linear_decay_replaced(),
    'cosine':        cosine_replaced(),
    'constant_0.5':  constant_replaced(0.5),
    'piecewise':     piecewise_demo(),
}

mask = editor.derive_mask(img, src_prompt, tgt_prompt, inversion_inner_steps=inner_used)

results = {}
for mode in ('none', 'attention'):
    for name, sched in SCHEDULES.items():
        key = f'{name}__{mode}'
        print(f'editing: {key}')
        results[key] = editor.edit(
            img, src_prompt, tgt_prompt,
            schedule=sched, mask_mode=mode,
            inversion_inner_steps=inner_used,
            precomputed_mask=mask if mode == 'attention' else None,
        )

In [ ]:
def grid(images, labels, n_cols, title=None):
    n_rows = (len(images) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3.3))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for i, ax in enumerate(axes):
        if i < len(images):
            ax.imshow(images[i]); ax.set_title(labels[i], fontsize=8)
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

imgs = [img] + [results[f'{n}__none'] for n in SCHEDULES] + [results[f'{n}__attention'] for n in SCHEDULES]
labels = ['source'] + [f'{n}\nno_mask' for n in SCHEDULES] + [f'{n}\nattention' for n in SCHEDULES]
grid(imgs, labels, n_cols=len(SCHEDULES) + 1, title=f'{entry["image"]} | {src_prompt!r} -> {tgt_prompt!r}')

## 5. Full ablation

Loop over every image in prompts.json. Gate on reconstruction. For each passing image,
run all 5 schedules x 2 mask modes, compute metrics, save a CSV and per-image grids.

**Timing:** pre-compute each image's mask once and pass it in to every edit (apples-to-apples
bg_lpips). A typical image: 1 recon check + 1 scout + 10 edits ≈ 2-3 minutes. Full 20-image
set ≈ 45-60 minutes.

In [ ]:
import csv

out_dir = Path('outputs/ablation_nb')
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / 'metrics.csv'

dropped = []
with csv_path.open('w', newline='') as fp:
    writer = csv.writer(fp)
    writer.writerow(['image', 'edit_type', 'schedule', 'mask_mode', 'clip_dir', 'bg_lpips', 'recon_lpips'])

    for entry in entries:
        img_path = real_dir / entry['image']
        if not img_path.exists():
            dropped.append((entry['image'], 'file missing')); continue
        img = Image.open(img_path).convert('RGB').resize((512, 512))
        src_prompt, tgt_prompt = entry['source'], entry['target']

        # Drop degenerate entries up front (no need to burn inversion time on them).
        if src_prompt == tgt_prompt:
            dropped.append((entry['image'], 'source == target (plural-substitution no-op)')); continue
        # "Mixed prompts" = descriptive source + bare target, a sign the BLIP caption didn't
        # contain the source word, so the COCO label was wrong and we fell back to a bare target.
        # These swaps have almost no structural overlap so they never edit well.
        if tgt_prompt.startswith('a photograph of a ') and not src_prompt.startswith('a photograph of a '):
            dropped.append((entry['image'], 'mixed prompts (COCO label mismatch, bare target fallback)')); continue

        print(f'\n=== {entry["image"]}  |  {src_prompt!r} -> {tgt_prompt!r}')
        lpips_score, passed, inner_used = reconstruct_and_gate(img, src_prompt)
        if not passed:
            dropped.append((entry['image'], f'recon LPIPS {lpips_score:.3f}'))
            print(f'  SKIP (recon too high)')
            continue

        mask = editor.derive_mask(img, src_prompt, tgt_prompt, inversion_inner_steps=inner_used)
        row_imgs, row_labels = [img], ['source']
        for sched_name, sched in SCHEDULES.items():
            for mode in ('none', 'attention'):
                edit = editor.edit(
                    img, src_prompt, tgt_prompt,
                    schedule=sched, mask_mode=mode,
                    inversion_inner_steps=inner_used,
                    precomputed_mask=mask if mode == 'attention' else None,
                )
                clip_dir = clip_directional_similarity(img, edit, src_prompt, tgt_prompt)
                bg = background_lpips(img, edit, mask)
                writer.writerow([entry['image'], entry.get('edit_type', ''), sched_name, mode,
                                 f'{clip_dir:.4f}', f'{bg:.4f}', f'{lpips_score:.4f}'])
                row_imgs.append(edit); row_labels.append(f'{sched_name}\n{mode}')
                print(f'  {sched_name:<14} {mode:<10} clip_dir={clip_dir:.4f} bg_lpips={bg:.4f}')

        # save per-image grid to disk
        fig, axes = plt.subplots(1, len(row_imgs), figsize=(len(row_imgs) * 2.5, 3))
        for ax, im, lab in zip(axes, row_imgs, row_labels):
            ax.imshow(im); ax.set_title(lab, fontsize=6); ax.axis('off')
        plt.tight_layout()
        plt.savefig(out_dir / f'{Path(entry["image"]).stem}_grid.png', dpi=100, bbox_inches='tight')
        plt.close(fig)

print(f'\ncsv: {csv_path}')
print(f'dropped {len(dropped)} images:')
for img_name, why in dropped:
    print(f'  {img_name}: {why}')

## 6. Summary tables for the report

Average metrics per (schedule, mask_mode) pair, split by edit_type. This is the
"schedule wins where" table the proposal calls for.

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
print(f'total edits: {len(df)}  |  unique images: {df.image.nunique()}')

print('\nmean CLIP-dir and bg_lpips by (schedule, mask_mode):')
agg = df.groupby(['schedule', 'mask_mode'])[['clip_dir', 'bg_lpips']].mean().round(4)
print(agg)

print('\nbroken down by edit_type:')
if 'edit_type' in df.columns and df.edit_type.notna().any():
    for et, sub in df.groupby('edit_type'):
        if not et: continue
        print(f'\n  {et}:')
        print(sub.groupby(['schedule', 'mask_mode'])[['clip_dir', 'bg_lpips']].mean().round(4))

In [ ]:
# the headline: which schedule beats vanilla P2P on CLIP-dir, with bg_lpips held down by the mask?
masked = df[df.mask_mode == 'attention']
per_sched = masked.groupby('schedule')[['clip_dir', 'bg_lpips']].mean().sort_values('clip_dir', ascending=False)
print('with attention mask, sorted by CLIP-dir (higher is better):')
print(per_sched.round(4))
vanilla_clip = per_sched.loc['vanilla_p2p', 'clip_dir']
for s, row in per_sched.iterrows():
    if row.clip_dir > vanilla_clip:
        print(f'  -> {s} beats vanilla_p2p by {row.clip_dir - vanilla_clip:+.4f} CLIP-dir')

## Troubleshooting

- **Everything skipped with "recon too high":** your prompts don't describe the images. Re-run
  `scripts/caption_real_photos.py` (wipes prompts.json with fresh BLIP captions) or edit
  `data/prompts.json` by hand for specific entries.
- **Mask is all white or all black:** check that source/target have at least one differing token
  (the `replaced` role). If they're identical, no swap happens and the mask degenerates.
- **CUDA OOM on the 4-way batched edit:** unlikely on A100 but possible on T4. Drop
  `num_inference_steps` to 40 and retry.
- **Ghost ears / cutout artifacts:** already defaulted to soft mask + late-step skip. If still
  visible, bump `mask_blend_until_frac` down (e.g. 0.5) to release the last 50% of steps.